# 02 — Feature Engineering → Lance `frames` Table Version Update

**Purpose:** Use Ray to compute CLIP embeddings for all frames in the Lance `frames` dataset and write them back as a new dataset version, preserving the prior version for rollback.

**What this notebook does:**

1. **Open the Lance dataset and pin the source version.** Read from the Lance `frames` dataset written in notebook 01. Pinning a version ensures that if new frames are appended concurrently, this FE pass operates on a consistent snapshot.

2. **Initialize Ray with GPU workers.** Load the CLIP vision encoder once per worker actor (not per task) using a Ray actor class. Loading the model once and reusing it across batches avoids repeated CUDA initialization overhead — a common source of GPU underutilization in distributed inference.

3. **Read frames in batches from Lance.** Workers call `ds.to_batches(columns=['frame_id', 'image'], batch_size=256)` to stream frame batches. Column projection (`columns=['frame_id', 'image']`) avoids loading unused columns (metadata, labels) during inference.

4. **Run CLIP inference per batch.** Each worker decodes JPEG bytes → PIL Image → CLIP preprocessor → vision encoder forward pass. Output is a `fixed_size_list<float32>[768]` embedding per frame.

5. **Write embeddings back as a new Lance version.** Merge `frame_id` + `embedding` results into a PyArrow table and use `lance.write_dataset(mode='merge', on='frame_id')` to update the embedding column in-place, creating a new dataset version. Version N+1 contains all original columns plus populated embeddings; version N is preserved and unchanged.

6. **Mosaic Vector Search (note).** The same embedding vectors can be pushed separately to a Databricks Mosaic Vector Search index for retrieval use cases. This is a distinct concern from the training data store — Lance holds embeddings as features for training; Vector Search holds them for ANN query serving.

7. **Verify.** Confirm the new dataset version exists, spot-check that embedding vectors are non-null and have the correct dimension, and print the version history.

**Inputs:** Lance `frames` dataset v1 at `/Volumes/{catalog}/{schema}/{volume}/frames/`

**Outputs:** Lance `frames` dataset v2 — same table, new version with `embedding` column populated.

**Next:** Run `03_train_model.ipynb` to use this versioned dataset for model training.